# Closed-loop adaptive GRAPE

This notebook keeps the same simplified style as `test_grape.ipynb` and `adaptive_grape_parameter_fit.ipynb`, but now combines the pieces into a small closed-loop calibration experiment.

The loop is:

1. optimize a pulse with GRAPE using the current calibrated model,
2. probe nearby pulses on the hidden true model with 500-shot binary measurements,
3. fit Hamiltonian parameters and a photon-selective readout model from all measured data,
4. repeat, always warm-starting from the previous optimized pulse and previous fitted parameters.

The first phase uses unitary evolution for GRAPE and fitting. The second phase continues from that result and uses non-unitary evolution for GRAPE and fitting, while still keeping `T1/T2` fixed rather than fitting them.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import optax
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

cwd = Path.cwd()
project_dir = cwd if (cwd / "grape.py").exists() else cwd / "Simplified_adaptive_grape"
repo_root = project_dir.parent
sys.path.insert(0, str(project_dir))
sys.path.insert(0, str(repo_root))

import toolbox as tbx
import grape


## Experimental values and hidden true model

The nominal model is what GRAPE believes. The hidden true model is what generates the fake experimental measurements. It is intentionally close to the nominal model, but has small frequency shifts, amplitude miscalibrations, and slightly shorter coherence times.

The measured value is not the true photon probability directly. The hidden experiment first computes the full photon-number distribution, then applies a photon-selective readout kernel. The resulting readout probability is sampled with a 500-shot binomial distribution. This mimics the long selective pulse plus qubit readout: the measurement mostly responds to photon number `n`, but it can have finite contrast, false positives, and weak leakage from neighboring photon numbers.


In [ ]:
config_path = repo_root / "configuration.json"
with config_path.open("r") as f:
    cfg = json.load(f)

# kHz -> MHz -> rad/us. We use a negative chi so H_disp = chi * n_cavity * |e><e|.
chi_nominal = -2 * jnp.pi * cfg["chi_kHz"] * 1e-3
cavity_self_kerr_nominal = 2 * jnp.pi * cfg["self_Kerr_kHz"] * 1e-3

qubit_T1_nominal = float(cfg["qubit_T1_us"])
qubit_T2_nominal = float(cfg["qubit_T2_us"])
cavity_T1_nominal = float(cfg["storage_T1_us"])
cavity_T2_nominal = float(cfg["storage_T2_us"])

mu_qub = 20.0
mu_cav = 20.0

# Hidden true model. Same soft mismatch as the parameter-fit notebook.
chi_true_shift_MHz = 0.001
qubit_freq_shift_MHz = 0.020
cavity_freq_shift_MHz = 0.040
cavity_self_kerr_true_MHz = -0.0007
qubit_amp_factor_true = 1.0005
cavity_amp_factor_true = 0.995

chi_true = chi_nominal + 2 * jnp.pi * chi_true_shift_MHz
qubit_shift_true = 2 * jnp.pi * qubit_freq_shift_MHz
cavity_shift_true = 2 * jnp.pi * cavity_freq_shift_MHz
cavity_self_kerr_true = 2 * jnp.pi * cavity_self_kerr_true_MHz

qubit_T1_true = 0.96 * qubit_T1_nominal
qubit_T2_true = 0.93 * qubit_T2_nominal
cavity_T1_true = 0.98 * cavity_T1_nominal
cavity_T2_true = 0.96 * cavity_T2_nominal

nominal_model = {
    "chi": chi_nominal,
    "qubit_shift": 0.0,
    "cavity_shift": 0.0,
    "cavity_self_kerr": cavity_self_kerr_nominal,
    "qubit_amp_factor": 1.0,
    "cavity_amp_factor": 1.0,
    "qubit_T1": qubit_T1_nominal,
    "qubit_T2": qubit_T2_nominal,
    "cavity_T1": cavity_T1_nominal,
    "cavity_T2": cavity_T2_nominal,
}

true_model = {
    "chi": chi_true,
    "qubit_shift": qubit_shift_true,
    "cavity_shift": cavity_shift_true,
    "cavity_self_kerr": cavity_self_kerr_true,
    "qubit_amp_factor": qubit_amp_factor_true,
    "cavity_amp_factor": cavity_amp_factor_true,
    "qubit_T1": qubit_T1_true,
    "qubit_T2": qubit_T2_true,
    "cavity_T1": cavity_T1_true,
    "cavity_T2": cavity_T2_true,
}

shots_per_pulse = 500

# Hidden photon-selective readout. The trained model has the same kernel family,
# but it starts from different parameters and does not know these true values.
true_response = {
    "q_dark": 0.010,
    "q_bright": 0.900,
    "delta": 0.10,
    "sigma": 0.45,
    "eps": 0.015,
    # Extra hidden offset: this makes the fake experiment slightly different
    # from the nominal readout initialization even before shot noise.
    "offset": 0.020,
}

print("nominal chi [rad/us]:", float(chi_nominal))
print("true chi shift [MHz]:", chi_true_shift_MHz)
print("true qubit/cavity shifts [MHz]:", qubit_freq_shift_MHz, cavity_freq_shift_MHz)
print("true readout response:", true_response)


## Pulse basis and simulation grid

The control is still `4 x 20` real B-spline coefficients. The two skipped splines at each end force every basis pulse to start and end at zero. With quadratic splines, at most three basis functions overlap at a given time.


In [ ]:
N_cav = 25
target_n = 2 #fock state to prepare

param_clip = 2.0
n_channels = 4
n_total_bsplines = 24
spline_degree = 2
skip_left = 2
skip_right = 2
bspln_num = n_total_bsplines - skip_left - skip_right
assert bspln_num == 20

T_us = 1.408
Nt = 130

time_start = 0.0
time_end = T_us
time_edges = jnp.linspace(time_start, time_end, Nt + 1)
time_mids = 0.5 * (time_edges[1:] + time_edges[:-1])
time_intervals = time_edges[1:] - time_edges[:-1]

bspline_builder = tbx.setup_bspline_builder(
    time_start,
    time_end,
    n_total_bsplines,
    spline_degree,
    skip_left,
    skip_right,
)
bsplns_mids = jnp.asarray(bspline_builder(np.asarray(time_mids)))
bsplns_edges = jnp.asarray(bspline_builder(np.asarray(time_edges)))

print("B-splines on midpoints:", bsplns_mids.shape)
print("endpoint max:", float(jnp.max(jnp.abs(bsplns_edges[:, [0, -1]]))))
print("max active B-splines:", int(jnp.max(jnp.sum(bsplns_mids > 1e-12, axis=0))))

plt.figure(figsize=(9, 3))
for b in np.asarray(bsplns_edges):
    plt.plot(np.asarray(time_edges) * 1e3, b, lw=1)
plt.xlabel("time [ns]")
plt.ylabel("basis value")
plt.title("20 quadratic B-splines after skipping two at each edge")
plt.grid(alpha=0.3)
plt.show()


## Operators and simulators


In [ ]:
a = tbx.tensor(tbx.identity(2), tbx.destroy(N_cav))
adag = tbx.hconj(a)
n_phot = adag @ a
n2_minus_n = n_phot @ n_phot - n_phot

sigz = tbx.tensor(tbx.sigma.z, tbx.identity(N_cav))
sigp = tbx.tensor(tbx.sigma.p, tbx.identity(N_cav))  # |e> -> |g>
sigm = tbx.hconj(sigp)                              # |g> -> |e>
one = tbx.identity(2 * N_cav)
qubit_excited = 0.5 * (one - sigz)

psi_init = tbx.tensor(tbx.basis(2, 0), tbx.basis(N_cav, 0))
rho_init = psi_init @ tbx.hconj(psi_init)

print("Hilbert dimension:", psi_init.shape[0])


In [ ]:
def hamiltonian_tree(ctrl_coeffs, model):
    e_qub, e_cav = grape.controls_from_coefficients(ctrl_coeffs, bsplns_mids)
    e_qub = model["qubit_amp_factor"] * e_qub
    e_cav = model["cavity_amp_factor"] * 1j * jnp.conj(e_cav)

    H_drift = (
        model["chi"] * (n_phot @ qubit_excited)
        + 0.5 * model["cavity_self_kerr"] * n2_minus_n
        + model["cavity_shift"] * n_phot
        + model["qubit_shift"] * qubit_excited
    )

    return [
        [H_drift, 1.0, 1.0, 0.0],
        [sigp, mu_qub * e_qub, 1.0, 1.0],
        [adag, mu_cav * e_cav, 1.0, 1.0],
    ]


def collapse_ops(model):
    gamma_phi_qub = jnp.maximum(1.0 / model["qubit_T2"] - 0.5 / model["qubit_T1"], 0.0)
    gamma_phi_cav = jnp.maximum(1.0 / model["cavity_T2"] - 0.5 / model["cavity_T1"], 0.0)

    return [
        jnp.sqrt(1.0 / model["qubit_T1"]) * sigp,
        jnp.sqrt(2.0 * gamma_phi_qub) * qubit_excited,
        jnp.sqrt(1.0 / model["cavity_T1"]) * a,
        jnp.sqrt(2.0 * gamma_phi_cav) * n_phot,
    ]


def photon_distribution_from_state(psi):
    psi = psi.reshape(2, N_cav)
    return jnp.sum(jnp.abs(psi) ** 2, axis=0).real


def photon_distribution_from_density(rho):
    rho = rho.reshape(2, N_cav, 2, N_cav)
    return jnp.real(jnp.diag(rho[0, :, 0, :]) + jnp.diag(rho[1, :, 1, :]))


def unitary_photon_distribution(ctrl_coeffs, model):
    psi_t = tbx.sesolve_htree(hamiltonian_tree(ctrl_coeffs, model), psi_init, time_intervals)
    return photon_distribution_from_state(psi_t[-1])


def decay_photon_distribution(ctrl_coeffs, model):
    rho_final = tbx.mesolve_htree(
        hamiltonian_tree(ctrl_coeffs, model),
        collapse_ops(model),
        rho_init,
        time_intervals,
    )
    return photon_distribution_from_density(rho_final)


def unitary_probability(ctrl_coeffs, model):
    return unitary_photon_distribution(ctrl_coeffs, model)[target_n]


def decay_probability(ctrl_coeffs, model):
    return decay_photon_distribution(ctrl_coeffs, model)[target_n]


def readout_selectivity(response):
    photon_numbers = jnp.arange(N_cav)
    center = target_n + response["delta"]
    width = jnp.maximum(response["sigma"], 1e-3)
    gaussian = jnp.exp(-0.5 * ((photon_numbers - center) / width) ** 2)
    return response["eps"] + (1.0 - response["eps"]) * gaussian


def readout_probability_from_distribution(photon_probs, response):
    selected_population = jnp.sum(readout_selectivity(response) * photon_probs)
    q = response["q_dark"] + (response["q_bright"] - response["q_dark"]) * selected_population
    q = q + response["offset"]
    return jnp.clip(q, 1e-6, 1.0 - 1e-6)


## True experimental measurement

The true experiment always uses the hidden non-unitary model. The calibration algorithm never sees `p_true` or the true readout parameters; it only sees finite-shot successes.

Here the readout probability is not `A * P_n + B`. It is computed from the full photon-number distribution:

`q_readout = q_dark + (q_bright - q_dark) * sum_m s_m P(m) + offset`

where `s_m` is a narrow Gaussian-like selectivity curve centered near the target photon number.


In [ ]:
@jax.jit
def true_photon_distribution(ctrl_coeffs):
    return decay_photon_distribution(ctrl_coeffs, true_model)


@jax.jit
def true_physical_probability(ctrl_coeffs):
    return true_photon_distribution(ctrl_coeffs)[target_n]


@jax.jit
def true_readout_probability(ctrl_coeffs):
    return readout_probability_from_distribution(true_photon_distribution(ctrl_coeffs), true_response)


def measure_true_experiment(ctrl_coeffs, key):
    p_true = true_physical_probability(ctrl_coeffs)
    q_true = true_readout_probability(ctrl_coeffs)
    # This is the fake experimental shot noise: k ~ Binomial(N_shots, q_true).
    successes = jax.random.binomial(key, n=shots_per_pulse, p=q_true)
    observed = successes / shots_per_pulse
    return observed, successes, p_true


## Calibrated model parameters

We fit Hamiltonian parameters and a small photon-selective readout model.

Hamiltonian parameters:

- chi shift,
- qubit frequency shift,
- cavity self-Kerr,
- cavity frequency shift,
- qubit amplitude factor,
- cavity amplitude factor.

Readout parameters:

- `q_dark`: false-positive floor,
- `q_bright`: bright-state readout probability,
- `delta`: offset of the selective pulse center in photon-number units,
- `sigma`: width of the selective pulse in photon-number units,
- `eps`: weak off-target leakage.

The hidden true readout also has a small extra offset. The fitted model does not get a separate offset knob; it has to absorb that mismatch as well as it can through the physical readout parameters.

We do **not** fit `T1/T2`; those stay fixed at the nominal values. In the unitary phase they are unused. In the non-unitary phase they are included in the simulator but not adjusted.


In [ ]:
def model_from_fit_raw(raw):
    chi_shift_MHz = 0.010 * jnp.tanh(raw[0])
    qubit_shift_MHz = 0.080 * jnp.tanh(raw[1])
    cavity_self_kerr_MHz = cfg["self_Kerr_kHz"] * 1e-3 + 0.0015 * jnp.tanh(raw[2])
    cavity_shift_MHz = 0.120 * jnp.tanh(raw[3])

    qubit_amp_factor = 1.0 + 0.010 * jnp.tanh(raw[4])
    cavity_amp_factor = 1.0 + 0.030 * jnp.tanh(raw[5])

    # Readout model. These are bounded so the fit cannot invent absurd readout
    # probabilities or a very broad/nonselective measurement.
    q_dark = jnp.clip(0.010 + 0.025 * jnp.tanh(raw[6]), 1e-4, 0.08)
    q_bright = jnp.clip(0.930 + 0.060 * jnp.tanh(raw[7]), q_dark + 1e-3, 0.995)
    delta = 0.45 * jnp.tanh(raw[8])
    sigma = 0.55 + 0.35 * jnp.tanh(raw[9])
    eps = jnp.clip(0.020 + 0.020 * jnp.tanh(raw[10]), 0.0, 0.08)

    model = {
        "chi": chi_nominal + 2 * jnp.pi * chi_shift_MHz,
        "qubit_shift": 2 * jnp.pi * qubit_shift_MHz,
        "cavity_shift": 2 * jnp.pi * cavity_shift_MHz,
        "cavity_self_kerr": 2 * jnp.pi * cavity_self_kerr_MHz,
        "qubit_amp_factor": qubit_amp_factor,
        "cavity_amp_factor": cavity_amp_factor,
        "qubit_T1": qubit_T1_nominal,
        "qubit_T2": qubit_T2_nominal,
        "cavity_T1": cavity_T1_nominal,
        "cavity_T2": cavity_T2_nominal,
    }
    response = {
        "q_dark": q_dark,
        "q_bright": q_bright,
        "delta": delta,
        "sigma": sigma,
        "eps": eps,
        "offset": 0.0,
    }
    return model, response


def predicted_observed_unitary(raw, ctrl_coeffs):
    model, response = model_from_fit_raw(raw)
    photon_probs = unitary_photon_distribution(ctrl_coeffs, model)
    return readout_probability_from_distribution(photon_probs, response)


def predicted_observed_decay(raw, ctrl_coeffs):
    model, response = model_from_fit_raw(raw)
    photon_probs = decay_photon_distribution(ctrl_coeffs, model)
    return readout_probability_from_distribution(photon_probs, response)


def parameter_summary(raw):
    model, response = model_from_fit_raw(raw)
    return {
        "chi_shift_MHz": float((model["chi"] - chi_nominal) / (2 * jnp.pi)),
        "qubit_shift_MHz": float(model["qubit_shift"] / (2 * jnp.pi)),
        "cavity_shift_MHz": float(model["cavity_shift"] / (2 * jnp.pi)),
        "self_kerr_MHz": float(model["cavity_self_kerr"] / (2 * jnp.pi)),
        "qubit_amp": float(model["qubit_amp_factor"]),
        "cavity_amp": float(model["cavity_amp_factor"]),
        "q_dark": float(response["q_dark"]),
        "q_bright": float(response["q_bright"]),
        "delta": float(response["delta"]),
        "sigma": float(response["sigma"]),
        "eps": float(response["eps"]),
    }


## Local probing around the optimized pulse

After GRAPE has produced a candidate pulse, we do not measure only that single pulse. One measurement point would tell us whether this pulse is good, but it would give very little information about *which Hamiltonian/readout parameters are wrong*. To calibrate a model, we need to see how the measured probability changes when the pulse is changed slightly.

So each round creates a small local cloud of controls around the current optimized pulse:

- the first control is exactly the optimized pulse,
- most controls get very small random coefficient perturbations,
- a smaller fraction get slightly larger perturbations,
- all coefficients are clipped back to the allowed `[-2, 2]` range.

Each control in this cloud is then sent to the hidden true experiment. For one control `u_i`, the experiment returns one scalar count:

`successes_i ~ Binomial(N_shots, q_true(u_i))`.

The arrays built in this cell therefore have one entry per tested pulse. For example, `successes_values[j]` is the number of successful readout shots for the `j`th nearby pulse. These new measurements are appended to the accumulated dataset, so later parameter fits use both old and new pulse/readout pairs.


In [ ]:
dataset_size_per_round = 500
very_local_noise_std = 0.0025
medium_local_noise_std = 0.0060
very_local_fraction = 0.90


def probe_local_dataset(ctrl_center, key, label):
    center_true = true_physical_probability(ctrl_center)
    center_expected_observed = true_readout_probability(ctrl_center)
    print(f"{label} center hidden true P_n: {float(center_true):.6f}")
    print(f"{label} center expected readout probability: {float(center_expected_observed):.6f}")

    n_very_local = int(dataset_size_per_round * very_local_fraction)
    noise_scales = jnp.concatenate([
        very_local_noise_std * jnp.ones((n_very_local,)),
        medium_local_noise_std * jnp.ones((dataset_size_per_round - n_very_local,)),
    ])

    key, noise_key, perm_key = jax.random.split(key, 3)
    noise_scales = noise_scales[jax.random.permutation(perm_key, dataset_size_per_round)]
    noise = noise_scales[:, None, None] * jax.random.normal(
        noise_key,
        (dataset_size_per_round, n_channels, bspln_num),
    )

    controls = grape.clip_coefficients(ctrl_center[None, :, :] + noise, param_clip)
    controls = controls.at[0].set(ctrl_center)

    observed_values = []
    true_values = []
    successes_values = []

    pbar = tqdm(range(dataset_size_per_round), desc=f"probe {label}", leave=False)
    for i in pbar:
        key, measure_key = jax.random.split(key)
        observed, successes, p_true = measure_true_experiment(controls[i], measure_key)
        observed_values.append(observed)
        true_values.append(p_true)
        successes_values.append(successes)
        pbar.set_postfix(obs=f"{float(observed):.4f}", true=f"{float(p_true):.4f}")

    observed_values = jnp.asarray(observed_values)
    true_values = jnp.asarray(true_values)
    successes_values = jnp.asarray(successes_values)

    print(f"{label} observed mean/std: {float(jnp.mean(observed_values)):.5f} / {float(jnp.std(observed_values)):.5f}")
    print(f"{label} true P_n mean/best: {float(jnp.mean(true_values)):.5f} / {float(jnp.max(true_values)):.5f}")
    return key, controls, observed_values, successes_values, true_values


def append_dataset(old_controls, old_observed, old_successes, old_true, new_controls, new_observed, new_successes, new_true):
    if old_controls is None:
        return new_controls, new_observed, new_successes, new_true
    return (
        jnp.concatenate([old_controls, new_controls], axis=0),
        jnp.concatenate([old_observed, new_observed], axis=0),
        jnp.concatenate([old_successes, new_successes], axis=0),
        jnp.concatenate([old_true, new_true], axis=0),
    )


## GRAPE steps

This is the pulse-optimization part. Here the fitted model parameters are held fixed, and only the B-spline coefficients of the pulse are updated.

For a fixed calibration vector `fit_raw`, `model_from_fit_raw(fit_raw)` builds the current Hamiltonian model. GRAPE then asks JAX to differentiate the final target Fock probability with respect to the control coefficients:

`control coefficients -> B-spline pulse -> time evolution -> P_model,n`.

The loss is

`loss = 1 - P_model,n + pulse_penalty`.

Minimizing this loss is equivalent to maximizing the model-predicted target Fock probability, while discouraging unnecessarily rough or large pulses through the penalty term.

There are two versions:

- `unitary_grape_step` uses the faster unitary simulator,
- `nonunitary_grape_step` uses the density-matrix simulator with decay/dephasing.

Both functions use `jax.value_and_grad(...)(ctrl_coeffs)`: the gradient is with respect to the 80 control coefficients, not with respect to the Hamiltonian parameters. After Adam applies the update, the coefficients are clipped to the allowed range. The hidden true model is never used inside GRAPE; it is only used afterward when we simulate experimental measurements.


In [ ]:
unitary_grape_steps_per_round = 250
unitary_grape_learning_rate = 0.020
unitary_grape_optimizer = optax.adam(unitary_grape_learning_rate)

nonunitary_grape_steps_per_round = 140
nonunitary_grape_learning_rate = 0.002
nonunitary_grape_optimizer = optax.adam(nonunitary_grape_learning_rate)


@jax.jit
def unitary_grape_step(ctrl_coeffs, opt_state, fit_raw):
    def loss_fn(c):
        model, _ = model_from_fit_raw(fit_raw)
        p = unitary_probability(c, model)
        return 1.0 - p + grape.pulse_penalty(c), p

    (loss_value, p_value), grads = jax.value_and_grad(loss_fn, has_aux=True)(ctrl_coeffs)
    updates, opt_state = unitary_grape_optimizer.update(grads, opt_state)
    ctrl_coeffs = optax.apply_updates(ctrl_coeffs, updates)
    ctrl_coeffs = grape.clip_coefficients(ctrl_coeffs, param_clip)
    return ctrl_coeffs, opt_state, loss_value, p_value


@jax.jit
def nonunitary_grape_step(ctrl_coeffs, opt_state, fit_raw):
    def loss_fn(c):
        model, _ = model_from_fit_raw(fit_raw)
        p = decay_probability(c, model)
        return 1.0 - p + grape.pulse_penalty(c), p

    (loss_value, p_value), grads = jax.value_and_grad(loss_fn, has_aux=True)(ctrl_coeffs)
    updates, opt_state = nonunitary_grape_optimizer.update(grads, opt_state)
    ctrl_coeffs = optax.apply_updates(ctrl_coeffs, updates)
    ctrl_coeffs = grape.clip_coefficients(ctrl_coeffs, param_clip)
    return ctrl_coeffs, opt_state, loss_value, p_value


def run_unitary_grape(ctrl_start, fit_raw, label):
    ctrl = ctrl_start
    opt_state = unitary_grape_optimizer.init(ctrl)
    history = []
    pbar = tqdm(range(unitary_grape_steps_per_round), desc=f"unitary GRAPE {label}", leave=False)
    for _ in pbar:
        ctrl, opt_state, loss_value, p_value = unitary_grape_step(ctrl, opt_state, fit_raw)
        history.append(float(p_value))
        pbar.set_postfix(P_n=f"{float(p_value):.5f}")
    return ctrl, history


def run_nonunitary_grape(ctrl_start, fit_raw, label):
    ctrl = ctrl_start
    opt_state = nonunitary_grape_optimizer.init(ctrl)
    history = []
    pbar = tqdm(range(nonunitary_grape_steps_per_round), desc=f"nonunitary GRAPE {label}", leave=False)
    for _ in pbar:
        ctrl, opt_state, loss_value, p_value = nonunitary_grape_step(ctrl, opt_state, fit_raw)
        history.append(float(p_value))
        pbar.set_postfix(P_n=f"{float(p_value):.5f}")
    return ctrl, history


## Parameter fitting steps

The fit sees only finite-shot readout data. For each measured pulse we have `successes = k` out of `N = shots_per_pulse` binary shots. The calibrated model predicts a success probability `q` for that same pulse.

`NLL` means **negative log likelihood**. It answers: *if the model predicted probability `q`, how surprising would it be to observe `k` successes?* For binomial shot data,

`k ~ Binomial(N, q)`

The probability of seeing exactly `k` successes is

`Pr(k | q, N) = C(N,k) q^k (1-q)^(N-k)`.

This has a simple interpretation: every successful shot contributes a factor `q`, every failed shot contributes a factor `(1-q)`, and `C(N,k)` counts how many orders of successes/failures could have produced the same total count. For fitting `q`, the combinatorial factor `C(N,k)` is irrelevant because it depends only on the measured data, not on the model prediction.

Taking the log turns the product into a sum:

`log Pr(k | q, N) = constant + k log(q) + (N-k) log(1-q)`.

Optimizers usually minimize losses, so we take the negative log likelihood and drop the constant:

`NLL(q; k, N) = -k log(q) - (N-k) log(1-q)`.

So the loss is smallest when the predicted `q` agrees with the observed fraction `k/N`, but it keeps the correct statistics for finite binary measurements. The division by `N` in the function below just keeps the loss scale comparable if we later change the number of shots.

Now the important part: this is also how the Hamiltonian parameters are trained.

The trainable vector is `fit_raw`. It is converted by `model_from_fit_raw` into physical parameters such as `chi`, detunings, self-Kerr, drive-amplitude scale factors, and readout-kernel parameters. For each measured pulse in a mini-batch, the code performs the following differentiable chain:

`fit_raw -> Hamiltonian/readout parameters -> simulated photon distribution -> predicted readout probability q -> binomial NLL`.

Then JAX computes

`d NLL / d fit_raw`.

So the Hamiltonian parameters are not trained by comparing them to the hidden true parameters. They are trained only because changing them changes the simulated evolution, which changes the predicted readout probability `q`, which changes how well the model explains the measured success counts.

The unitary phase uses `predicted_observed_unitary`; the non-unitary phase uses `predicted_observed_decay`. In both cases, the same Adam optimizer updates Hamiltonian parameters and readout-kernel parameters together, using mini-batches sampled from all accumulated measurements.


In [ ]:
fit_steps_per_round = 180
fit_batch_size = 8
fit_learning_rate = 0.020
fit_optimizer = optax.adam(fit_learning_rate)


def binomial_nll_from_successes(q, successes):
    q = jnp.clip(q, 1e-6, 1.0 - 1e-6)
    failures = shots_per_pulse - successes
    # Divide by shots_per_pulse so the scale is comparable when changing shots.
    return -(successes * jnp.log(q) + failures * jnp.log1p(-q)) / shots_per_pulse


@jax.jit
def fit_step_unitary(fit_raw, opt_state, batch_controls, batch_successes):
    def loss_fn(raw):
        pred = jax.vmap(lambda c: predicted_observed_unitary(raw, c))(batch_controls)
        return jnp.mean(binomial_nll_from_successes(pred, batch_successes)), pred

    (loss_value, pred), grads = jax.value_and_grad(loss_fn, has_aux=True)(fit_raw)
    updates, opt_state = fit_optimizer.update(grads, opt_state)
    fit_raw = optax.apply_updates(fit_raw, updates)
    return fit_raw, opt_state, loss_value, jnp.mean(pred)


@jax.jit
def fit_step_decay(fit_raw, opt_state, batch_controls, batch_successes):
    def loss_fn(raw):
        pred = jax.vmap(lambda c: predicted_observed_decay(raw, c))(batch_controls)
        return jnp.mean(binomial_nll_from_successes(pred, batch_successes)), pred

    (loss_value, pred), grads = jax.value_and_grad(loss_fn, has_aux=True)(fit_raw)
    updates, opt_state = fit_optimizer.update(grads, opt_state)
    fit_raw = optax.apply_updates(fit_raw, updates)
    return fit_raw, opt_state, loss_value, jnp.mean(pred)


def fit_parameters(fit_raw, controls_seen, successes_seen, key, mode, label):
    opt_state = fit_optimizer.init(fit_raw)
    losses = []
    n_seen = controls_seen.shape[0]

    pbar = tqdm(range(fit_steps_per_round), desc=f"fit {mode} model {label}", leave=False)
    for _ in pbar:
        key, batch_key = jax.random.split(key)
        batch_idx = jax.random.choice(batch_key, n_seen, shape=(fit_batch_size,), replace=False)
        if mode == "unitary":
            fit_raw, opt_state, loss_value, mean_pred = fit_step_unitary(
                fit_raw,
                opt_state,
                controls_seen[batch_idx],
                successes_seen[batch_idx],
            )
        else:
            fit_raw, opt_state, loss_value, mean_pred = fit_step_decay(
                fit_raw,
                opt_state,
                controls_seen[batch_idx],
                successes_seen[batch_idx],
            )
        losses.append(float(loss_value))
        pbar.set_postfix(nll=f"{float(loss_value):.3e}", pred=f"{float(mean_pred):.4f}")

    print(f"{label} fit NLL first/last: {losses[0]:.4e} / {losses[-1]:.4e}")
    print(f"{label} fitted parameters:", parameter_summary(fit_raw))
    return key, fit_raw, losses


## Initial pulse and shared state


In [ ]:
key = jax.random.key(123)
key, subkey = jax.random.split(key)

initial_coeffs = 0.03 * jax.random.normal(subkey, (n_channels, bspln_num))
initial_coeffs = grape.clip_coefficients(initial_coeffs, param_clip)

fit_raw = jnp.zeros((11,))
controls_seen = None
observed_seen = None
successes_seen = None
true_seen = None

print("initial true P_n:", float(true_physical_probability(initial_coeffs)))
print("initial true readout probability:", float(true_readout_probability(initial_coeffs)))
print("initial fitted parameters:", parameter_summary(fit_raw))


## Phase 1: closed-loop unitary adaptive GRAPE

Each round runs unitary GRAPE, measures local pulses on the hidden true open-system model, then refits Hamiltonian and measurement-response parameters using unitary evolution.


In [ ]:
n_unitary_rounds = 20
ctrl_current = initial_coeffs
unitary_records = []
unitary_grape_histories = []
unitary_fit_histories = []

for round_index in range(n_unitary_rounds):
    label = f"unitary round {round_index:02d}"
    print("\n" + "=" * 80)
    print(label)
    print("starting parameters:", parameter_summary(fit_raw))

    ctrl_current, grape_history = run_unitary_grape(ctrl_current, fit_raw, label)
    unitary_grape_histories.append(grape_history)

    fitted_model, fitted_response = model_from_fit_raw(fit_raw)
    predicted_p = unitary_probability(ctrl_current, fitted_model)
    predicted_observed = predicted_observed_unitary(fit_raw, ctrl_current)
    hidden_true_p = true_physical_probability(ctrl_current)
    expected_observed = true_readout_probability(ctrl_current)
    print(f"{label} GRAPE predicted unitary P_n: {float(predicted_p):.6f}")
    print(f"{label} model predicted readout probability: {float(predicted_observed):.6f}")
    print(f"{label} hidden true P_n: {float(hidden_true_p):.6f}")
    print(f"{label} hidden expected readout probability: {float(expected_observed):.6f}")

    key, new_controls, new_observed, new_successes, new_true = probe_local_dataset(ctrl_current, key, label)
    controls_seen, observed_seen, successes_seen, true_seen = append_dataset(
        controls_seen,
        observed_seen,
        successes_seen,
        true_seen,
        new_controls,
        new_observed,
        new_successes,
        new_true,
    )
    print(f"total measured controls: {controls_seen.shape[0]}")

    key, fit_raw, fit_losses = fit_parameters(
        fit_raw,
        controls_seen,
        successes_seen,
        key,
        mode="unitary",
        label=label,
    )
    unitary_fit_histories.append(fit_losses)

    unitary_records.append({
        "round": round_index,
        "predicted_p": float(predicted_p),
        "predicted_observed": float(predicted_observed),
        "true_p": float(hidden_true_p),
        "expected_observed": float(expected_observed),
        "center_measured_observed": float(new_observed[0]),
        "data_observed_mean": float(jnp.mean(new_observed)),
        "data_true_mean": float(jnp.mean(new_true)),
        "fit_loss_last": float(fit_losses[-1]),
    })


## Phase 2: closed-loop non-unitary adaptive GRAPE

This phase starts from the unitary result. It uses non-unitary GRAPE and non-unitary parameter fitting, but the fitted parameters are still only Hamiltonian and measurement-response parameters. The nominal `T1/T2` values are used but not fitted.


In [ ]:
n_nonunitary_rounds = 20
nonunitary_records = []
nonunitary_grape_histories = []
nonunitary_fit_histories = []

for round_index in range(n_nonunitary_rounds):
    label = f"nonunitary round {round_index:02d}"
    print("\n" + "=" * 80)
    print(label)
    print("starting parameters:", parameter_summary(fit_raw))

    ctrl_current, grape_history = run_nonunitary_grape(ctrl_current, fit_raw, label)
    nonunitary_grape_histories.append(grape_history)

    fitted_model, fitted_response = model_from_fit_raw(fit_raw)
    predicted_p = decay_probability(ctrl_current, fitted_model)
    predicted_observed = predicted_observed_decay(fit_raw, ctrl_current)
    hidden_true_p = true_physical_probability(ctrl_current)
    expected_observed = true_readout_probability(ctrl_current)
    print(f"{label} GRAPE predicted decay P_n: {float(predicted_p):.6f}")
    print(f"{label} model predicted readout probability: {float(predicted_observed):.6f}")
    print(f"{label} hidden true P_n: {float(hidden_true_p):.6f}")
    print(f"{label} hidden expected readout probability: {float(expected_observed):.6f}")

    key, new_controls, new_observed, new_successes, new_true = probe_local_dataset(ctrl_current, key, label)
    controls_seen, observed_seen, successes_seen, true_seen = append_dataset(
        controls_seen,
        observed_seen,
        successes_seen,
        true_seen,
        new_controls,
        new_observed,
        new_successes,
        new_true,
    )
    print(f"total measured controls: {controls_seen.shape[0]}")

    key, fit_raw, fit_losses = fit_parameters(
        fit_raw,
        controls_seen,
        successes_seen,
        key,
        mode="decay",
        label=label,
    )
    nonunitary_fit_histories.append(fit_losses)

    nonunitary_records.append({
        "round": round_index,
        "predicted_p": float(predicted_p),
        "predicted_observed": float(predicted_observed),
        "true_p": float(hidden_true_p),
        "expected_observed": float(expected_observed),
        "center_measured_observed": float(new_observed[0]),
        "data_observed_mean": float(jnp.mean(new_observed)),
        "data_true_mean": float(jnp.mean(new_true)),
        "fit_loss_last": float(fit_losses[-1]),
    })


## Diagnostics

The next plots separate two quantities that should not be mixed.

First, we compare the **measured readout probability** of the optimized pulse: the model predicts a photon-selective readout probability `q_readout`, while the hidden experiment returns a 500-shot measured fraction whose expectation is the hidden `q_true`.

Second, we compare the underlying **physical photon probability** `P_n`. This is a diagnostic quantity in simulation, not directly what the experiment reports. When `P_n` is close to 1, the last plot uses `1 - P_n` on a log scale so improvements near high fidelity are visible.


In [ ]:
from matplotlib.ticker import MaxNLocator


def record_array(records, name, fallback=None):
    values = []
    for r in records:
        if name in r:
            values.append(r[name])
        elif fallback is not None:
            values.append(fallback(r))
        else:
            values.append(np.nan)
    return np.asarray(values, dtype=float)


def predicted_measurement(records):
    return record_array(records, "predicted_observed")


def plot_measurement_comparison(ax, records, title):
    rounds = np.arange(len(records))
    model_meas = predicted_measurement(records)
    measured_center = record_array(
        records,
        "center_measured_observed",
        fallback=lambda r: r["expected_observed"],
    )
    expected_true = record_array(records, "expected_observed")

    ax.plot(rounds, model_meas, "o-", label="model predicted readout probability")
    ax.plot(rounds, measured_center, "s-", label="simulated 500-shot measured fraction")
    ax.plot(rounds, expected_true, "k--", lw=1.2, label="hidden expected readout probability")
    ax.set_title(title)
    ax.set_xlabel("closed-loop round")
    ax.set_ylabel("readout probability / measured fraction")
    ax.set_ylim(0.0, 1.0)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)


def plot_probability_comparison(ax, records, title):
    rounds = np.arange(len(records))
    ax.plot(rounds, record_array(records, "predicted_p"), "o-", label=f"model predicted P_{target_n}")
    ax.plot(rounds, record_array(records, "true_p"), "s-", label=f"hidden true P_{target_n}")
    ax.plot(rounds, record_array(records, "data_true_mean"), ".-", alpha=0.55, label="mean true P_n of local probes")
    ax.set_title(title)
    ax.set_xlabel("closed-loop round")
    ax.set_ylabel(f"P_{target_n}")
    ax.set_ylim(0.0, 1.02)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)


fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
plot_measurement_comparison(axes[0], unitary_records, "Optimization with unitary evolution")
plot_measurement_comparison(axes[1], nonunitary_records, "Optimization with non-unitary evolution")
fig.suptitle("Optimized pulse: model readout prediction vs simulated measurement")
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
plot_probability_comparison(axes[0], unitary_records, "Optimization with unitary evolution")
plot_probability_comparison(axes[1], nonunitary_records, "Optimization with non-unitary evolution")
fig.suptitle("Optimized pulse: model photon probability vs hidden true photon probability")
fig.tight_layout()
plt.show()

all_records = unitary_records + nonunitary_records
rounds = np.arange(len(all_records))
true_infidelity = np.maximum(1.0 - record_array(all_records, "true_p"), 1e-8)
model_infidelity = np.maximum(1.0 - record_array(all_records, "predicted_p"), 1e-8)

plt.figure(figsize=(9, 4))
plt.semilogy(rounds, true_infidelity, "s-", label=f"hidden true 1 - P_{target_n}")
plt.semilogy(rounds, model_infidelity, "o-", label=f"model predicted 1 - P_{target_n}")
if len(unitary_records) > 0 and len(nonunitary_records) > 0:
    boundary = len(unitary_records) - 0.5
    plt.axvline(boundary, color="0.35", ls="--", lw=1)
    ymax = plt.ylim()[1]
    plt.text(boundary, ymax, " switch to non-unitary optimization", va="top", ha="left", fontsize=9)
plt.xlabel("global closed-loop round")
plt.ylabel(f"infidelity 1 - P_{target_n}")
plt.title("Optimized pulse quality on a log infidelity scale")
plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
plt.grid(alpha=0.3, which="both")
plt.legend()
plt.tight_layout()
plt.show()

print("final fitted parameters:", parameter_summary(fit_raw))
print("final hidden true P_n:", float(true_physical_probability(ctrl_current)))
print("final hidden expected readout probability:", float(true_readout_probability(ctrl_current)))


# Dynamiqs double check

In [ ]:
import dynamiqs as dq

# Independent dynamiqs mesolve check for the final optimized pulse ctrl_current.
# This uses a denser 4 ns grid and rebuilds the Hamiltonian independently from
# the toolbox solver used during GRAPE.

dt_check_ns = 4
T_ns = int(round(T_us * 1000))
t_edges_check_us = jnp.arange(0, T_ns + dt_check_ns, dt_check_ns) / 1000.0
t_mids_check_us = 0.5 * (t_edges_check_us[1:] + t_edges_check_us[:-1])
bsplns_check = jnp.asarray(bspline_builder(np.asarray(t_mids_check_us)))

Iq = dq.eye(2)
Ic = dq.eye(N_cav)

a_q = dq.tensor(dq.destroy(2), Ic)
a_c = dq.tensor(Iq, dq.destroy(N_cav))
a_q_dag = dq.dag(a_q)
a_c_dag = dq.dag(a_c)

n_q = a_q_dag @ a_q
n_c = a_c_dag @ a_c

psi0_dq = dq.tensor(dq.basis(2, 0), dq.basis(N_cav, 0))
rho0_dq = dq.todm(psi0_dq)
projector_n = dq.tensor(Iq, dq.todm(dq.basis(N_cav, target_n)))
projectors_m = [dq.tensor(Iq, dq.todm(dq.basis(N_cav, m))) for m in range(N_cav)]

method = dq.method.Tsit5(rtol=1e-8, atol=1e-10, max_steps=1_000_000)
options = dq.Options(save_states=True, progress_meter=True)



def dynamiqs_mesolve_probability(ctrl_coeffs, model, label):
    fields = jnp.asarray(ctrl_coeffs @ bsplns_check)

    eps_qub_I = model["qubit_amp_factor"] * fields[0]
    eps_qub_Q = model["qubit_amp_factor"] * fields[1]
    eps_cav_I = model["cavity_amp_factor"] * fields[2]
    eps_cav_Q = model["cavity_amp_factor"] * fields[3]

    H0_dq = (
        model["chi"] * (n_c @ n_q)
        + 0.5 * model["cavity_self_kerr"] * (a_c_dag @ a_c_dag @ a_c @ a_c)
        + model["cavity_shift"] * n_c
        + model["qubit_shift"] * n_q
    )

    H_qub_I = mu_qub * (a_q + a_q_dag)
    H_qub_Q = mu_qub * (1j * a_q - 1j * a_q_dag)

    H_cav_I = mu_cav * (1j * a_c_dag - 1j * a_c)
    H_cav_Q = mu_cav * (a_c_dag + a_c)

    gamma_phi_qub = jnp.maximum(1.0 / model["qubit_T2"] - 0.5 / model["qubit_T1"], 0.0)
    gamma_phi_cav = jnp.maximum(1.0 / model["cavity_T2"] - 0.5 / model["cavity_T1"], 0.0)

    jump_ops_dq = [
        jnp.sqrt(1.0 / model["qubit_T1"]) * a_q,
        jnp.sqrt(2.0 * gamma_phi_qub) * n_q,
        jnp.sqrt(1.0 / model["cavity_T1"]) * a_c,
        jnp.sqrt(2.0 * gamma_phi_cav) * n_c,
    ]

    H_t = (
        H0_dq
        + dq.pwc(t_edges_check_us, eps_qub_I, H_qub_I)
        + dq.pwc(t_edges_check_us, eps_qub_Q, H_qub_Q)
        + dq.pwc(t_edges_check_us, eps_cav_I, H_cav_I)
        + dq.pwc(t_edges_check_us, eps_cav_Q, H_cav_Q)
    )

    result = dq.mesolve(
        H=H_t,
        jump_ops=jump_ops_dq,
        rho0=rho0_dq,
        tsave=t_edges_check_us,
        exp_ops=[projector_n],
        method=method,
        options=options,
    )

    rho_final = result.states[-1]
    photon_probs = jnp.asarray([jnp.real(dq.expect(projector, rho_final)) for projector in projectors_m])
    p_n = float(photon_probs[target_n])
    print(f"{label}: dynamiqs mesolve P_{target_n} = {p_n:.8f}")
    return p_n, photon_probs, result



fitted_model, fitted_response = model_from_fit_raw(fit_raw)

p_fitted_dq, photon_probs_fitted_dq, result_fitted_dq = dynamiqs_mesolve_probability(
    ctrl_current,
    fitted_model,
    "final pulse on fitted calibrated model",
)

p_true_dq, photon_probs_true_dq, result_true_dq = dynamiqs_mesolve_probability(
    ctrl_current,
    true_model,
    "final pulse on hidden true model",
)

print()
print("toolbox fitted-model decay P_n:", float(decay_probability(ctrl_current, fitted_model)))
print("toolbox hidden-true decay P_n:", float(true_physical_probability(ctrl_current)))
print("dynamiqs fitted-model readout probability:", float(readout_probability_from_distribution(photon_probs_fitted_dq, fitted_response)))
print("dynamiqs hidden-true readout probability:", float(readout_probability_from_distribution(photon_probs_true_dq, true_response)))
